# 01 – EDA: LinkedIn job postings

Goal: decide the project scope with numbers.

1. How many data / AI postings are there?
2. How many of them have a salary?
3. How are salaries spread across states?
4. How many postings per role?
5. How long are the job descriptions?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

DATA_DIR = "../data/raw/linkedin"
postings = pd.read_csv(f"{DATA_DIR}/postings.csv")
postings.shape

(123849, 31)

In [2]:
postings.info()

<class 'pandas.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  str    
 2   title                       123849 non-null  str    
 3   description                 123842 non-null  str    
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   str    
 6   location                    123849 non-null  str    
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  str    
 12  applies                     23320 non-null   float64
 13  original_listed_time     

In [3]:
missing = postings.isna().mean().sort_values(ascending=False)
missing.round(3)

closed_time                   0.991
skills_desc                   0.980
med_salary                    0.949
remote_allowed                0.877
applies                       0.812
max_salary                    0.759
min_salary                    0.759
currency                      0.709
compensation_type             0.709
normalized_salary             0.709
pay_period                    0.709
posting_domain                0.323
application_url               0.296
formatted_experience_level    0.237
fips                          0.221
zip_code                      0.169
company_name                  0.014
company_id                    0.014
views                         0.014
description                   0.000
work_type                     0.000
sponsored                     0.000
job_id                        0.000
listed_time                   0.000
expiry                        0.000
application_type              0.000
original_listed_time          0.000
formatted_work_type         

In [4]:
salary_cols = ["min_salary", "med_salary", "max_salary", "pay_period",
               "currency", "compensation_type", "normalized_salary"]
postings[salary_cols].sample(10, random_state=42)

,min_salary,med_salary,max_salary,pay_period,currency,compensation_type,normalized_salary
73989,NaN,NaN,NaN,NaN,NaN,NaN,NaN
59308,19.75,NaN,19.75,HOURLY,USD,BASE_SALARY,41080.0
44663,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81954,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113151,30.00,NaN,35.00,HOURLY,USD,BASE_SALARY,67600.0
60072,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50425,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75853,35.00,NaN,40.00,HOURLY,USD,BASE_SALARY,78000.0
72947,NaN,NaN,NaN,NaN,NaN,NaN,NaN
58062,NaN,72.0,NaN,HOURLY,USD,BASE_SALARY,149760.0


In [5]:
print(postings["pay_period"].value_counts(dropna=False))
print(postings["currency"].value_counts(dropna=False))
print(postings["remote_allowed"].value_counts(dropna=False))

pay_period
NaN         87776
YEARLY      20628
HOURLY      14741
MONTHLY       518
WEEKLY        177
BIWEEKLY        9
Name: count, dtype: int64
currency
NaN    87776
USD    36058
EUR        6
CAD        3
BBD        2
AUD        2
GBP        2
Name: count, dtype: int64
remote_allowed
NaN    108603
1.0     15246
Name: count, dtype: int64


In [6]:
postings.groupby("pay_period")["normalized_salary"].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
pay_period,,,,,,,,
BIWEEKLY,9.0,1248662.0,908696.0,53482.0,72943.0,1572805.0,1897389.0,2096744.0
HOURLY,14741.0,330624.0,7927651.0,14560.0,40622.0,52520.0,86320.0,535600000.0
MONTHLY,518.0,64083.0,74190.0,420.0,28347.0,53634.0,82109.0,1200000.0
WEEKLY,177.0,111560.0,21382.0,50804.0,99684.0,109148.0,119658.0,210548.0
YEARLY,20628.0,119685.0,715898.0,0.0,72500.0,102000.0,142500.0,102500000.0


In [7]:
sal = postings[postings["normalized_salary"].notna()].copy()
print("with salary:", len(sal))

sal = sal[(sal["currency"] == "USD") & (sal["pay_period"] != "BIWEEKLY")]
print("USD, no BIWEEKLY:", len(sal))

LOW, HIGH = 20_000, 600_000
outliers = sal[(sal["normalized_salary"] < LOW) | (sal["normalized_salary"] > HIGH)]
print("outside range:", len(outliers))

cols = ["title", "pay_period", "min_salary", "med_salary", "max_salary", "normalized_salary"]
outliers[cols].sort_values("normalized_salary").head(10)

with salary: 36073
USD, no BIWEEKLY: 36049
outside range: 534


,title,pay_period,min_salary,med_salary,max_salary,normalized_salary
71180,Community Manager,YEARLY,NaN,0.0,NaN,0.0
8305,Full-Stack Engineer Intern (Unpaid),YEARLY,NaN,0.0,NaN,0.0
17594,Sales Consultant (B2B),YEARLY,NaN,0.0,NaN,0.0
6395,Independent Sales Representative,YEARLY,NaN,0.0,NaN,0.0
62013,Public Health Empowerment Pro Bono Coach,YEARLY,NaN,0.0,NaN,0.0
46750,UX/UI Designer,YEARLY,NaN,0.0,NaN,0.0
62083,Ikigai-Inspired Coach – CrowdDoing Pro Bono Coaching Collective,YEARLY,NaN,0.0,NaN,0.0
25681,Marketing Specialist,YEARLY,NaN,0.0,NaN,0.0
60721,"Volunteer Scrum Master – Driving Nature Connection and Public Health, CrowdDoing.world's Nature ...",YEARLY,NaN,0.0,NaN,0.0
62061,Environmental Stewardship Pro Bono Coach,YEARLY,NaN,0.0,NaN,0.0


In [8]:
outliers[cols].sort_values("normalized_salary").tail(10)

,title,pay_period,min_salary,med_salary,max_salary,normalized_salary
87241,Full Time Field Service Technician – Anaheim/Pomona Area,HOURLY,62000.0,NaN,68000.0,135200000.0
17646,DCS Technician,HOURLY,70000.0,NaN,80000.0,156000000.0
72585,Trial Paralegal - Hybrid in OC or LA!,HOURLY,80000.0,NaN,100000.0,187200000.0
71845,Sharepoint Developer,HOURLY,100000.0,NaN,120000.0,228800000.0
102481,Manager - Clinical Applications Support - Ancillary Systems,HOURLY,98467.0,NaN,123094.0,230423440.0
60038,Billing Supervisor (246521),HOURLY,125000.0,NaN,125000.0,260000000.0
13614,Cloud Domain Architect,HOURLY,125000.0,NaN,150000.0,286000000.0
89082,Cloud Domain Architect,HOURLY,125000.0,NaN,150000.0,286000000.0
98888,"Case Manager RN, Pedi Rheumatology",HOURLY,151900.0,NaN,196570.0,362408800.0
9237,Intellectual Property Associate (246215),HOURLY,240000.0,NaN,275000.0,535600000.0


In [9]:
sal["raw_mid"] = sal["med_salary"].fillna((sal["min_salary"] + sal["max_salary"]) / 2)

wrong_hourly = (sal["pay_period"] == "HOURLY") & (sal["raw_mid"] >= 1_000)
print("hourly that look yearly:", wrong_hourly.sum())

sal.loc[wrong_hourly, "normalized_salary"] = sal.loc[wrong_hourly, "raw_mid"]
sal.loc[wrong_hourly, "pay_period"] = "YEARLY"
sal["salary_fixed"] = wrong_hourly

out = (sal["normalized_salary"] < LOW) | (sal["normalized_salary"] > HIGH)
print("still outside range:", out.sum())
sal[out].groupby("pay_period")["normalized_salary"].describe().round(0)

hourly that look yearly: 29
still outside range: 511


,count,mean,std,min,25%,50%,75%,max
pay_period,,,,,,,,
HOURLY,8.0,246436.0,457118.0,14560.0,15938.0,17451.0,170040.0,1248000.0
MONTHLY,28.0,86420.0,282569.0,420.0,6450.0,12000.0,13500.0,1200000.0
YEARLY,475.0,258373.0,4704259.0,0.0,26.0,52.0,125.0,102500000.0


In [10]:
wrong_yearly = (sal["pay_period"] == "YEARLY") & sal["raw_mid"].between(10, 200)
print("yearly that look hourly:", wrong_yearly.sum())

sal.loc[wrong_yearly, "normalized_salary"] = sal.loc[wrong_yearly, "raw_mid"] * 2080
sal.loc[wrong_yearly, "pay_period"] = "HOURLY"
sal["salary_fixed"] = sal["salary_fixed"] | wrong_yearly

out = (sal["normalized_salary"] < LOW) | (sal["normalized_salary"] > HIGH)
print("dropped:", out.sum())
sal = sal[~out].copy()

print("clean salary rows:", len(sal))
print("fixed rows:", sal["salary_fixed"].sum())
sal["normalized_salary"].describe().round(0)

yearly that look hourly: 372
dropped: 139
clean salary rows: 35910
fixed rows: 395


count     35910.0
mean      96398.0
std       58275.0
min       20000.0
25%       52260.0
50%       82500.0
75%      125000.0
max      600000.0
Name: normalized_salary, dtype: float64

### Salary cleaning rules

- Keep only USD, drop BIWEEKLY (9 rows, broken values).
- HOURLY with raw value >= 1,000 → really yearly, use raw value.
- YEARLY with raw value between 10 and 200 → really hourly, multiply by 2,080.
- Drop everything still outside 20,000 – 600,000 USD per year.
- `salary_fixed` marks every repaired row.